This environment comes pre-installed with qiskit v 2.3.0

# Imports 

* some may not be needed here since I am ot running on hardware

In [ ]:
# This should be obvious
from qiskit import QuantumCircuit


# This creates a sparse representation of the Pauli (or Paulis)
from qiskit.quantum_info import SparsePauliOp


# This is used to convert a circuit to the actual gates 
# used by the actual qc to be used
from qiskit.transpiler import generate_preset_pass_manager


# This is the extra code you need to run in an IBM qc or simulator
from qiskit_ibm_runtime import QiskitRuntimeService

# This is the new jargon in qiskit v2
from qiskit_ibm_runtime import EstimatorOptions
from qiskit_ibm_runtime import EstimatorV2 as Estimator

# Plotting capability
from matplotlib import pyplot as plt

In [ ]:
# Here we define a simulator
from qiskit_ibm_runtime.fake_provider import FakeBelemV2

# Here we create the quantum circuit

In [ ]:
# Create a new circuit with two qubits
qc = QuantumCircuit(2)
 
# Add a Hadamard gate to qubit 0
qc.h(0)
 
# Perform a controlled-X gate on qubit 1, controlled by qubit 0
qc.cx(0, 1)
 
qc.draw("mpl")

### We will define the observables (operators). In this snipet we define six different operators. So $IZ$ maps to $I \otimes Z$ (tensor product of two operators since we have 2 qubits)

In [ ]:
# Set up six different observables.
 
observables_labels = ["IZ", "IX", "ZI", "XI", "ZZ", "XX"]
observables = [SparsePauliOp(label) for label in observables_labels]

In [ ]:
observables

# Here I define a backend that simulates actual hardware

In [ ]:
# Here we define a fake computer since I have no access
backend = FakeBelemV2()
estimator = Estimator(backend)

# Convert to an ISA circuit 
## So this converts the circuit to the actual gates, that in this case a fake computer , the qc will implement

In [ ]:
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

In [ ]:
# And here it seems to me that each observable is "applied" to the isa circuit
mapped_observables = [
    observable.apply_layout(isa_circuit.layout) for observable in observables]

In [ ]:
# What is in the square bracket is the PUB (something something something), 
# as far as I can tell it is a generic object that represents a whole problem.
# Get used to it, it is used a lot

job = estimator.run([(isa_circuit, mapped_observables)])
result = job.result()

In [ ]:
# This is the result of the entire submission.  You submitted one Pub,
# so this contains one inner result (and some metadata of its own).
 
job_result = job.result()
 
# This is the result from our single pub, which had five observables,
# so contains information on all five.
 
pub_result = job.result()[0]

In [ ]:
len(job.result())

In [ ]:
# Plot the result
 
values = pub_result.data.evs
 
errors = pub_result.data.stds
 
# plotting graph
plt.plot(observables_labels, values, "-o")
plt.xlabel("Observables")
plt.ylabel("Values")
plt.show()

## Success!!